### Core Imports

In [ ]:
import sys, subprocess, shutil, platform, os, time
from pathlib import Path
import psutil, torch, re, stat
from types import SimpleNamespace


---
### Step 1 — Runtime check logic

Verifies GPU model, RAM, disk, and PyTorch/CUDA availability.

In [ ]:
def check_runtime():
    import sys, subprocess, shutil, platform
    import psutil, torch

    def _section(title):
        print(f"\n{'='*10} {title} {'='*10}")

    _section("System")
    print(f"Python   : {sys.version.split()[0]}")
    print(f"OS       : {platform.system()} {platform.release()}")

    _section("GPU")
    try:
        gpu = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
            capture_output=True, text=True
        )
        if gpu.returncode == 0 and gpu.stdout.strip():
            print(f"GPU      : {gpu.stdout.strip()}")
            print(subprocess.check_output('nvidia-smi').decode())
        else:
            print("GPU      : NOT FOUND — select A100 GPU runtime")
    except FileNotFoundError:
        print("GPU      : nvidia-smi not found")

    if torch.cuda.is_available():
        print(f"PyTorch  : {torch.__version__} | CUDA {torch.version.cuda}")
        print(f"GPU mem  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("PyTorch CUDA: not available")

    _section("RAM")
    vm = psutil.virtual_memory()
    ram_gb = vm.total / 1e9
    print(f"Total    : {ram_gb:.1f} GB  ({'OK' if ram_gb >= 50 else 'Low — select High-RAM runtime'})")
    print(f"Free     : {vm.available / 1e9:.1f} GB")

    _section("Disk")
    du = shutil.disk_usage('/')
    print(f"Free     : {du.free / 1e9:.1f} GB  ({'OK' if du.free / 1e9 >= 20 else 'Low'})")
    print(f"Total    : {du.total / 1e9:.1f} GB")

    _section("CPU")
    print(f"CPUs     : {psutil.cpu_count()}")

---
### Step 2 — Dependencies logic



In [ ]:
def install_dependencies():
    import subprocess, sys

    pkgs = [
        'affine', 'dask[distributed]', 'fiona', 'pyproj',
        'rasterio', 'rioxarray', 'shapely', 'stackstac', 'xarray',
        'pystac-client', 'pystac', 'planetary-computer', 'gdown', 'tqdm'
    ]

    print("Checking core dependencies...")
    try:
        print(f"Installing: {', '.join(pkgs)}...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
        print("Installation completed successfully.")
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"pip installation failed: {e}")
    except subprocess.CalledProcessError as e:
        print(f"[ERROR] An error occurred during pip installation. Please check the output for details. Error: {e}")
    except ImportError as e:
        print(f"[ERROR] Failed to import a core package even after installation attempt. This might indicate a problem with the installed package or the environment. Specific error: {e}")
    except Exception as e:
        print(f"[ERROR] An unexpected error occurred during dependency checking/installation: {e}")

---
### Step 3 — Path setup logic

In [1]:
def setup_paths(in_colab: bool) -> SimpleNamespace:
    """
    Configure all project paths.
    Drive must already be mounted before calling this (handled by master).
    Returns a SimpleNamespace so callers use paths.DATA_DIR etc.
    """
    from pathlib import Path
    from types import SimpleNamespace

    if in_colab:
        drive_base = Path('/content/drive/MyDrive/Colab Notebooks/TESSERA/Cocoa_Detection_Pipeline')
        repo_dir   = Path('/content/drive/MyDrive/Colab Notebooks/TESSERA/REPO')
        temp_dir   = Path('/content/tmp_tessera')
    else:
        drive_base = Path('./local_drive/TESSERA/Cocoa_Detection_Pipeline')
        repo_dir   = Path('./local_drive/TESSERA/REPO')
        temp_dir   = Path('./tmp_tessera')

    data_dir   = drive_base / 'data'
    checkpoint = drive_base / 'checkpoints'
    output_dir = drive_base / 'output'

    for d in [drive_base, data_dir, temp_dir, checkpoint, output_dir]:
        d.mkdir(parents=True, exist_ok=True)

    # acquisition_dates is a dict to check the S2 availability, for S1 we don't
    # need the check, as S2 is activily filtered and S1 is taken from source
    # without imposing a condition.
    paths = SimpleNamespace(
        DRIVE_BASE = drive_base,
        REPO_DIR   = repo_dir,
        DATA_DIR   = data_dir,
        TEMP_DIR   = temp_dir,
        CHECKPOINT = checkpoint,
        OUTPUT_DIR = output_dir,
        acquisition_dates = {}

    )

    print('Paths configured:')
    for name, path in [
        ('Drive base', paths.DRIVE_BASE),
        ('Repo',       paths.REPO_DIR),
        ('Data',       paths.DATA_DIR),
        ('Temp',       paths.TEMP_DIR),
        ('Checkpoint', paths.CHECKPOINT),
        ('Output',     paths.OUTPUT_DIR),
        ('Dates',     paths.acquisition_dates),
    ]:
        print(f'  {name:<12}: {path}')

    return paths


# ── Call it ───────────────────────────────────────────────────────────
# paths = setup_paths(in_colab=IN_COLAB)

NameError: name 'SimpleNamespace' is not defined

---
### Step 4 — Clone TESSERA repository logic

In [ ]:
def clone_tessera_repo(repo_dir: Path,
                       FORCE_RECLONE: bool = False,
                       BRANCH: str = 'alpha_version_1.0'):

    if FORCE_RECLONE and repo_dir.exists():
        print(f"Removing existing repo at: {repo_dir}")
        shutil.rmtree(repo_dir)

    if not repo_dir.exists():
        print(f"Cloning branch '{BRANCH}' to: {repo_dir}")
        subprocess.run([
            'git', 'clone', '--branch', BRANCH, '--single-branch',
            'https://github.com/ucam-eo/tessera.git', str(repo_dir)
        ], check=True)
        print("Clone complete.")
    else:
        print(f"Repo already present at: {repo_dir}")
        os.chdir(str(repo_dir))
        result = subprocess.run(['git', 'branch', '--show-current'], capture_output=True, text=True)
        print(f"Active branch: {result.stdout.strip()}")
        os.chdir('/content' if 'google.colab' in sys.modules else '.')

    # Add repo to Python path
    if str(repo_dir) not in sys.path:
        sys.path.append(str(repo_dir))
    print(f"REPO_DIR in sys.path: {str(repo_dir) in sys.path}")

---
### Step 5 — SCL patch logic
Adds class 10 (thin cirrus) to SCL_INVALID definition.

In [ ]:
def apply_scl_patch(repo_dir: Path) -> None:
    """
    Patch SCL_INVALID in s2_fast_processor.py to include class 10 (thin cirrus).
    """
    s2_proc = repo_dir / 'tessera_preprocessing' / 's2_fast_processor.py'

    if not s2_proc.exists():
        raise FileNotFoundError(f"s2_fast_processor.py not found at {s2_proc}")

    content = s2_proc.read_text()
    match = re.search(r'SCL_INVALID\s*=\s*\{([^}]*)\}', content)
    if not match:
        raise RuntimeError("Could not find SCL_INVALID in s2_fast_processor.py")

    current_values = match.group(1)
    print(f"SCL_INVALID before patch: {{{current_values.strip()}}}")

    if '10' in current_values:
        print("SCL patch already applied (class 10 present).")
    else:
        patched = re.sub(
            r'(SCL_INVALID\s*=\s*\{)([^}]*)(\})',
            lambda m: m.group(1) + m.group(2).rstrip() + ', 10' + m.group(3),
            content
        )
        s2_proc.write_text(patched)
        print("SCL patch applied — class 10 (thin cirrus) added.")

    for line in s2_proc.read_text().splitlines():
        if 'SCL_INVALID' in line:
            print(f"Verified: {line.strip()}")
            break


---
### Step 6 — Permissions logic

In [ ]:
def set_permissions(repo_dir: Path) -> None:
    """
    Set executable permissions on shell scripts and binary targets in the repo.
    """
    targets = (
        list((repo_dir / 'tessera_preprocessing').glob('*.sh')) +
        list((repo_dir / 'tessera_infer').glob('*.sh')) +
        [(repo_dir / 'tessera_preprocessing' / 's1_stack'),
         (repo_dir / 'tessera_preprocessing' / 's2_stack')]
    )

    print("Setting executable permissions:")
    for f in targets:
        if f.exists():
            f.chmod(f.stat().st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
            print(f"  ✓  {f.relative_to(repo_dir)}")
        else:
            print(f"  ✗  {f.relative_to(repo_dir)} (not found)")




---
### Step 7 — Inference Setup logic

Copying the checkpoint to the runtime is only necessary once per session. The checkpoint coud be read from any path, but copying it to the colab environment ensures that the right file is selected and that there are no spaces in the name or path.

In [ ]:
def setup_inference(repo_dir: Path,
                    checkpoint_dir: Path,
                    in_colab: bool,
                    ckpt_filename: str = 'best_model_fsdp_20250427_084307.pt') -> None:
    """
    Localize the checkpoint to /content/ for faster loading,
    and patch infer_all_tiles.sh with the current Python executable.
    """
    ckpt_local = Path('/content/checkpoint.pt') if in_colab else checkpoint_dir / ckpt_filename

    # 1. Localize checkpoint
    if ckpt_local.exists():
        print(f"Checkpoint already local: {ckpt_local} ({ckpt_local.stat().st_size / 1e6:.0f} MB)")
    else:
        if in_colab:
            print(f"Searching Drive for '{ckpt_filename}'...")
            matches = list(Path('/content/drive/MyDrive').rglob(ckpt_filename))
            if not matches:
                print(f"[WARNING] '{ckpt_filename}' not found on Drive. Falling back to CHECKPOINT dir.")
                src = checkpoint_dir / ckpt_filename
            else:
                src = matches[0]

            if src.exists():
                print(f"  Found: {src}")
                print("  Copying to /content/ for faster load (no spaces)...")
                shutil.copy2(src, ckpt_local)
                print(f"  Done — {ckpt_local.stat().st_size / 1e6:.0f} MB")
            else:
                raise FileNotFoundError(f"Source checkpoint not found at {src}")
        else:
            print("Local environment: skipping Drive localization.")

    # 2. Patch infer_all_tiles.sh
    script = repo_dir / 'tessera_infer' / 'infer_all_tiles.sh'
    if script.exists():
        content = script.read_text()
        old = 'export PYTHON_ENV="/absolute/path/to/your/python_env/bin/python"'
        new = f'export PYTHON_ENV="{sys.executable}"'
        if old in content:
            script.write_text(content.replace(old, new))
            print(f"PYTHON_ENV patched → {sys.executable}")
        else:
            print("PYTHON_ENV already patched.")
    else:
        raise FileNotFoundError(f"Script not found: {script}")

    return ckpt_local




### Pipeline Execution Engine

---
### Step 8 — ROI Generation logic

In [ ]:
def worker_step_roi(area_name: str, params: dict, paths: SimpleNamespace,
                    verbose=False):
    """
    Generates a ROI TIFF based on central coordinates and tile size.
    """
    utm_crs = CRS.from_epsg(params['epsg'])
    transformer = Transformer.from_crs('EPSG:4326', utm_crs, always_xy=True)
    cx, cy = transformer.transform(params['lon'], params['lat'])

    half = params['size_m'] / 2
    n_cells = int(params['size_m'] / params['res'])
    minx, miny, maxx, maxy = cx - half, cy - half, cx + half, cy + half

    output_path = paths.DATA_DIR / f"{area_name}_roi.tif"

    with rasterio.open(
        output_path, 'w',
        driver='GTiff', dtype='uint8', count=1,
        width=n_cells, height=n_cells,
        crs=utm_crs,
        transform=from_bounds(minx, miny, maxx, maxy, n_cells, n_cells)
    ) as dst:
        dst.write(np.ones((n_cells, n_cells), dtype=np.uint8), 1)

    print(f"ROI Generated: {output_path}")

    # Verify
    if verbose:
        with rasterio.open(output_path) as src:
            data = src.read(1)
            #print(f"ROI path   : {output_path}")
            print(f"Size       : {src.width}×{src.height} px")
            print(f"CRS        : {src.crs}")
            print(f"Resolution : {src.res}")
            print(f"Bounds     : {src.bounds}")
            print(f"Centre UTM : ({cx:.2f}, {cy:.2f})")
            print(f"Mask unique: {np.unique(data)}  (expected [1])")


    return output_path

---
### Step 9 — S1/S2 Download logic

The block is split into two functions as the file handling is not the same between the two methods.

**Download Sentinel-2 data (test tile, 2023)**

Expected output (as tested before, with the same criteria): ~77 tiffs (11 bands × 7 dates) in `/content/s2_download_output/`.

**Note:**  
`--max_cloud 20` gives sparse coverage — confirmed behaviour for this region.  
`--temp_dir` IS supported by `s2_fast_processor.py` (unlike S1).

**Download Sentinel-1 data (test tile, 2023)**

Expected output: 60 tiffs, as result from the previous test (VV + VH × 30 dates, ascending only — no descending orbit available on MPC for this region).  

**Note:**
`--temp_dir` is NOT supported by `s1_fast_processor.py` — omitted deliberately.

In [ ]:
def query_s2_dates(roi_path: Path, year: int, max_cloud: int = 20) -> list:
    """
    Query MPC STAC catalogue for available S2 dates without downloading.
    Returns a sorted list of date strings.
    """
    import pystac_client
    import planetary_computer

    # Get bounding box from ROI
    with rasterio.open(roi_path) as src:
        bounds = src.bounds
        crs = src.crs

    # Reproject to WGS84 for STAC query
    from pyproj import Transformer
    transformer = Transformer.from_crs(crs, 'EPSG:4326', always_xy=True)
    minx, miny = transformer.transform(bounds.left, bounds.bottom)
    maxx, maxy = transformer.transform(bounds.right, bounds.top)

    catalog = pystac_client.Client.open(
        'https://planetarycomputer.microsoft.com/api/stac/v1',
        modifier=planetary_computer.sign_inplace
    )

    search = catalog.search(
        collections=['sentinel-2-l2a'],
        bbox=[minx, miny, maxx, maxy],
        datetime=f'{year}-01-01/{year}-12-31',
        query={'eo:cloud_cover': {'lt': max_cloud}}
    )

    dates = sorted(set(
        item.datetime.strftime('%Y-%m-%d')
        for item in search.items()
    ))
    return dates

In [4]:
from datetime import datetime, timedelta

def _progress(processed, skipped, total, date, status, start_time):
    done = processed + skipped
    elapsed = time.time() - start_time
    rate = elapsed / done if done > 0 else 0
    remaining = (total - done) * rate
    eta = datetime.now() + timedelta(seconds=remaining)
    icon = '✓' if status == 'cached' else '⏱' if status == 'timeout' else '✗'
    print(f"  {icon} {date} ({status})  [{done}/{total}]  ETA: {eta.strftime('%H:%M:%S')}")


def worker_step_download_s2(roi_path: Path, year: int, paths: SimpleNamespace,
                            verbose=False, max_cloud=20, chunksize: int = 1024) -> Path:
    """
    Downloads Sentinel-2 data for the ROI and year.
    Uses paths.TEMP_DIR for high-speed I/O.
    """
    s2_out = paths.TEMP_DIR / 's2_download_output'
    s2_tmp = paths.TEMP_DIR / 's2_download_temp'
    for d in [s2_out, s2_tmp]:
        d.mkdir(parents=True, exist_ok=True)

    # Get dates for this threshold
    dates = query_s2_dates(roi_path, year, max_cloud=max_cloud)
    print(f"\n[S2] Starting download for {year} ({len(dates)} scenes at {max_cloud}% threshold)...")

    t0 = time.time()
    total_processed = 0
    total_skipped   = 0
    total_timeouts  = 0

    for i, date in enumerate(dates, 1):
        dt    = datetime.strptime(date, '%Y-%m-%d')
        start = (dt - timedelta(days=1)).strftime('%Y-%m-%d')
        end   = (dt + timedelta(days=1)).strftime('%Y-%m-%d')

        cmd = [
            sys.executable,
            str(paths.REPO_DIR / 'tessera_preprocessing' / 's2_fast_processor.py'),
            '--input_tiff',    str(roi_path),
            '--output',        str(s2_out),
            '--temp_dir',      str(s2_tmp),
            '--start_date',    start,
            '--end_date',      end,
            '--max_cloud',     '100',
            '--resolution',    '10',
            '--worker_memory', '12',
            '--dask_workers',  '2',
            '--chunksize',     str(chunksize),
            '--data_source',   'mpc',
        ]

        print(f"  [{i}/{len(dates)}] {date}", end='  ')

        for attempt in range(2):
            proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                                    stderr=subprocess.STDOUT, text=True)

            status = 'unknown'

            for line in proc.stdout:
                #print(line, end='')  # temporary — see all output
                if status != 'unknown':
                    continue
                if 'All bands already exist' in line:
                    status = 'cached'
                elif 'all bands processed successfully' in line:
                    status = 'cached'
                elif 'valid coverage' in line and '< 5.0%' in line:
                    status = 'skipped'
                elif 'SCL processing timeout' in line:
                    status = 'timeout'

            proc.wait()

            if status == 'cached':
                break  # success, no retry needed
            elif attempt == 0 and status in ['skipped', 'timeout', 'unknown']:
                print(f"↺ retry", end='  ')
            else:
                print(f"✗ failed", end='  ')
                break  # failed twice, move on

        if status == 'cached':
            total_processed += 1
            print(f"✓ cached")
        elif status == 'timeout':
            total_timeouts += 1
            print(f"⏱ timeout")
        else:
            total_skipped += 1
            print(f"✗ skipped")

        if proc.returncode != 0 and status == 'unknown':
            raise RuntimeError(f"S2 download failed on {date} with rc={proc.returncode}")

    if verbose:
        tiffs = list(s2_out.rglob('*.tiff'))
        bands = sorted(set(t.parent.name for t in tiffs))
        dates_found = sorted(set(t.stem.split('_')[0] for t in tiffs
                                  if t.parent.name != s2_out.name))
        print(f"Tiffs found : {len(tiffs)} across {len(dates_found)} dates, {len(bands)} bands")

    return s2_out


def worker_step_download_s1(roi_path: Path, year: int, paths: SimpleNamespace,
                            verbose=False)-> Path:
    """
    Downloads Sentinel-1 data for the ROI and year.
    Uses paths.TEMP_DIR for high-speed I/O.
    Note: --temp_dir is not a recognised argument for s1_fast_processor.py.
    """
    s1_out = paths.TEMP_DIR / 's1_download_output'
    s1_out.mkdir(parents=True, exist_ok=True)

    print(f"\n[S1] Starting download for {year}...")
    cmd = [
        sys.executable,
        str(paths.REPO_DIR / 'tessera_preprocessing' / 's1_fast_processor.py'),
        '--input_tiff',    str(roi_path),
        '--output',        str(s1_out),
        '--start_date',    f'{year}-01-01',
        '--end_date',      f'{year}-12-31',
        '--orbit_state',   'both',
        '--resolution',    '10',
        '--worker_memory', '4',
        '--dask_workers',  '8',
        '--min_coverage',  '50',
        '--data_source',   'mpc',
    ]
    t0 = time.time()
    subprocess.run(cmd, check=True)
    print(f"[S1] Done in {(time.time()-t0)/60:.1f} min")

    # Verify
    if verbose:
        tiffs  = sorted(s1_out.glob('*.tiff'))
        dates  = sorted(set(f.name.split('_')[0] for f in tiffs))
        orbits = sorted(set(f.stem.split('_')[-1] for f in tiffs))

        print(f"Tiffs found : {len(tiffs)} across {len(dates)} unique dates")
        print(f"Dates       : {len(dates)} unique  ({dates[0]} → {dates[-1]})")
        print(f"Orbit states: {orbits}")
        print("\nFirst 6 files:")
        for f in tiffs[:6]:
            print(f"  {f.name}")

    return s1_out

NameError: name 'Path' is not defined

---
### Step 10 — Stacking logic

Runs `s1_stack` and `s2_stack` in parallel (Python threads) against the downloaded tiffs.

| Binary | Input | Expected output |
|--------|-------|-----------------|
| `s1_stack` | `/content/s1_download_output/` (flat, n tiffs) | `.npy` SAR stack |
| `s2_stack` | `/content/s2_download_output/` (band subfolders) | `.npy` optical stack |

Both write into the same `STACKED_DIR`. `--rate 1` keeps every pixel (no downsampling).
Output is ephemeral (`/content/`) — copy to Drive in Cell 11 on success.

In [ ]:
def worker_step_stack(s1_dir: Path, s2_dir: Path, paths: SimpleNamespace,
                      verbose=False)-> Path:
    """
    Runs the Rust stackers in parallel to consolidate downloaded tiffs into .npy stacks.
    """
    stacked_out = paths.TEMP_DIR / 'stacked_output'
    stacked_out.mkdir(parents=True, exist_ok=True)

    preproc_dir = paths.REPO_DIR / 'tessera_preprocessing'
    n_parallel  = 8

    cmd_s1 = [
        str(preproc_dir / 's1_stack'),
        '--input-dir',  str(s1_dir),
        '--output-dir', str(stacked_out),
        '--parallel',   str(n_parallel),
        '--rate',       '1',
    ]

    cmd_s2 = [
        str(preproc_dir / 's2_stack'),
        '--input',        str(s2_dir),
        '--output',       str(stacked_out),
        '--batch-size',   '8',
        '--cache-level',  '1',
        '--num-threads',  str(n_parallel),
        '--sample-rate',  '1',
    ]

    results = {}
    def _run(name, cmd):
        t0 = time.time()
        proc = subprocess.run(cmd, capture_output=True, text=True)
        results[name] = {
            'rc':      proc.returncode,
            'elapsed': time.time() - t0,
            'stdout':  proc.stdout,
            'stderr':  proc.stderr,
        }

    print("\n[Stacking] Launching s1_stack and s2_stack in parallel...")
    t1 = threading.Thread(target=_run, args=('s1_stack', cmd_s1))
    t2 = threading.Thread(target=_run, args=('s2_stack', cmd_s2))
    t1.start(); t2.start()
    t1.join();  t2.join()
    print("Both processes finished.\n")

    # Report
    for name, r in results.items():
        status = '✓' if r['rc'] == 0 else '✗'
        print(f"  {status} {name}  (rc={r['rc']}, {r['elapsed']:.1f}s)")
        if r['stdout'].strip():
            print(r['stdout'][-2000:])
        if r['rc'] != 0:
            print(f"    STDERR: {r['stderr'][-1000:]}")

    # Verify
    if verbose:
        npy_files = sorted(stacked_out.glob('*.npy'))
        if not npy_files:
            print("  (no .npy files found — check stderr above)")
            return stacked_out

        print("\nStacked output:")
        for f in npy_files:
            arr = np.load(f)
            print(f"  {f.name:<30}  shape={str(arr.shape):<25}  dtype={arr.dtype}  {f.stat().st_size / 1e6:.1f} MB")

    return stacked_out

---
### Step 11 — Retiling logic → 40×40 patches (`dpixel_retiler.py`)

Slices the stacked output into 40×40 px patches matching TESSERA's `sample_size_s2/s1`.
A 500×500 tile produces a 12×12 grid (144 patches); edge patches are zero-padded.

| Argument | Value | Notes |
|----------|-------|-------|
| `--tiff_path` | `test_roi_5km.tif` | Reference CRS + transform |
| `--d_pixel_dir` | `/content/stacked_output` | All `.npy` files |
| `--patch_size` | `40` | Must match `sample_size_s2/s1` in config |
| `--out_dir` | `/content/retiled_d_pixel` | Matches `infer_all_tiles.sh` default name |
| `--block_size` | `500` | Whole tile fits in one block |
| `--num_workers` | `8` | Within 12-CPU budget |

In [ ]:
def worker_step_retile(roi_path: Path, stacked_dir: Path, paths: SimpleNamespace,
                       patch_size: int = 40, verbose=False) -> Path:
    """
    Slices the stacked .npy files into small patches for inference.
    """
    retiled_out = paths.TEMP_DIR / 'retiled_d_pixel'
    retiled_out.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        str(paths.REPO_DIR / 'tessera_preprocessing' / 'dpixel_retiler.py'),
        '--tiff_path',    str(roi_path),
        '--d_pixel_dir',  str(stacked_dir),
        '--patch_size',   str(patch_size),
        '--out_dir',      str(retiled_out),
        '--block_size',   '500',
        '--num_workers',  '8',
        '--overwrite',
    ]

    print(f"\n[Retiling] Slicing stacks into {patch_size}x{patch_size} patches...")
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - t0
    status = '✓' if proc.returncode == 0 else '✗'
    print(f"  {status}  rc={proc.returncode}  ({elapsed:.1f}s)")

    if proc.stdout.strip():
        print(proc.stdout[-3000:])
    if proc.returncode != 0:
        raise RuntimeError(f"dpixel_retiler.py failed:\n{proc.stderr[-2000:]}")

    # Verify
    if verbose:
        patch_dirs = sorted(retiled_out.iterdir())
        print(f"\nRetiled output structure:")
        print(f"  Patch directories : {len(patch_dirs)}")
        if patch_dirs:
            sample = patch_dirs[0]
            print(f"  Sample patch      : {sample.name}")
            for f in sorted(sample.iterdir()):
                print(f"    {f.name:<35}  {f.stat().st_size / 1e3:.1f} KB")

    return retiled_out

---
### Step 12 —  TESSERA inference (`infer_all_tiles.sh`)

Runs inference on the 169 retiled patches using the A100 GPU.
Checkpoint is read directly from Drive — no copying needed.

**Pre-run checklist:**
- Cell 10 completed — `/content/stacked_output/` populated
- Cell 11 completed — `/content/retiled_d_pixel/` contains 169 patch directories
- Checkpoint confirmed at `DRIVE_BASE/checkpoints/best_model_fsdp_20250427_084307.pt`
- `infer_all_tiles.sh` patched: `PYTHON_ENV` set to `/usr/bin/python3`

The patch is applied here before launch. It is idempotent — safe to re-run.

| Argument | Value |
|----------|-------|
| `--tiles-dir` | `/content/retiled_d_pixel` |
| `--output-dir` | `/content/representation_retiled` |
| `--checkpoint` | `DRIVE_BASE/checkpoints/best_model_fsdp_20250427_084307.pt` |
| `--cpu-gpu-split` | `0:1` (GPU only — A100) |

> **Note:** `infer_all_tiles.sh` backgrounds inference via `nohup`. The subprocess returns quickly (~1 min setup) while inference runs asynchronously. Output appears in `/content/representation_retiled/` and logs in `tessera_infer/logs/`.

In [ ]:
def worker_step_inference(retiled_dir: Path,
                          paths: SimpleNamespace,
                          ckpt_local: Path,
                          verbose=False)-> Path:
    """
    Executes TESSERA model inference on the retiled patches.
    Requires localized checkpoint from setup_inference.
    """
    repr_out = paths.TEMP_DIR / 'representation_retiled'
    repr_out.mkdir(parents=True, exist_ok=True)

    if not ckpt_local.exists():
        raise FileNotFoundError(f"Localized checkpoint not found at {ckpt_local}. Run setup_inference first.")

    infer_dir = paths.REPO_DIR / 'tessera_infer'

    cmd = [
        'bash', 'infer_all_tiles.sh',
        '--tiles-dir',     str(retiled_dir),
        '--output-dir',    str(repr_out),
        '--checkpoint',    str(ckpt_local),
        '--cpu-gpu-split', '0:1',
    ]

    print(f"\n[Inference] Running TESSERA model...")
    print(f"  Checkpoint : {ckpt_local}")
    print(f"  Tiles dir  : {retiled_dir}")
    print(f"  Output dir : {repr_out}\n")
    print(f"  ⏳ Long process — estimated 10 minutes, no output until complete...\n")

    t0 = time.time()
    proc = subprocess.run(
        cmd,
        capture_output=True, text=True,
        cwd=str(infer_dir),
        env={**os.environ, 'PYTHONPATH': str(paths.REPO_DIR)}
    )
    elapsed = time.time() - t0
    status = '✓' if proc.returncode == 0 else '✗'
    print(f"  {status}  rc={proc.returncode}  ({elapsed/60:.1f} min)")

    if proc.stdout.strip():
        print(proc.stdout[-3000:])
    if proc.returncode != 0:
        raise RuntimeError(f"Inference failed:\n{proc.stderr[-2000:]}")

    # Verify
    if verbose:
        repr_files = sorted(repr_out.rglob('*.npy'))
        print(f"\nRepresentation output:")
        print(f"  .npy files found : {len(repr_files)}")
        if repr_files:
            sample = repr_files[0]
            arr = np.load(sample)
            print(f"  Sample file      : {sample.name}")
            print(f"  Shape            : {arr.shape}  (expected (*,128))")

    return repr_out

---
### Step 13 — Stitching logic → (500, 500, 128) GeoTIFF (`stitch_tiled_representation.py`)

Reassembles the 169 × (40, 40, 128) embedding patches into a single spatially-referenced
output, cropped to the downstream TIFF extent.

In [ ]:
def worker_step_stitch(roi_path: Path, retiled_dir: Path, repr_dir: Path,
                       paths: SimpleNamespace, verbose=False) -> Path:
    """
    Reassembles the patch embeddings into a single spatially-referenced .npy stack.
    """
    stitched_out = paths.TEMP_DIR / 'stitched_output'
    stitched_out.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        str(paths.REPO_DIR / 'tessera_infer' / 'stitch_tiled_representation.py'),
        '--d_pixel_retiled_path',        str(retiled_dir),
        '--representation_retiled_path', str(repr_dir),
        '--downstream_tiff',             str(roi_path),
        '--out_dir',                     str(stitched_out),
    ]

    print(f"\n[Stitching] Reassembling patches into a single .npy stack...")
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - t0
    status = '✓' if proc.returncode == 0 else '✗'
    print(f"  {status}  rc={proc.returncode}  ({elapsed:.1f}s)")

    if proc.stdout.strip():
        print(proc.stdout[-3000:])
    if proc.returncode != 0:
        raise RuntimeError(f"Stitching failed:\n{proc.stderr[-2000:]}")

    # Verify
    if verbose:
        print("\nStitched output:")
        for f in sorted(stitched_out.iterdir()):
            print(f"  {f.name:<40}  {f.stat().st_size / 1e6:.1f} MB")
            if f.suffix == '.npy':
                arr = np.load(f)
                print(f"    shape : {arr.shape}  (expected (500, 500, 128))")
            elif f.suffix in ('.tif', '.tiff'):
                with rasterio.open(f) as src:
                    print(f"    size  : {src.width}×{src.height}  bands={src.count}  crs={src.crs}")

    return stitched_out

---
### Cell 14 — Convert stitched embedding to GeoTIFF and save to Drive

Converts `stitched_representation.npy` (500, 500, 128) to a 128-band GeoTIFF
using `test_roi_5km.tif` as the spatial reference, then copies to Drive.

In [ ]:
def worker_step_export(area_name: str, year: int, roi_path: Path,
                       stitched_dir: Path, paths: SimpleNamespace,
                       verbose=False, max_cloud=20)-> Path:
    """
    Converts the stitched representation into a GeoTIFF and copies it to Drive.
    """
    import io
    sys.path.insert(0, str(paths.REPO_DIR / 'tessera_infer'))
    from convert_npy2tiff import convert_npy_to_tiff

    stitched_npy = stitched_dir / 'stitched_representation.npy'
    if not stitched_npy.exists():
        raise FileNotFoundError(f"Stitched .npy not found at {stitched_npy}")

    # Convert — suppress verbose band-by-band output
    print(f"\n[Export] Converting stitched_representation.npy → GeoTIFF...")
    t0 = time.time()
    _stdout = sys.stdout
    sys.stdout = io.StringIO()
    convert_npy_to_tiff(
        npy_path        = str(stitched_npy),
        ref_tiff_path   = str(roi_path),
        out_dir         = str(stitched_dir),
        downsample_rate = 1,
    )
    sys.stdout = _stdout
    print(f"  Done in {time.time()-t0:.1f}s")

    # Verify local output
    local_tif = stitched_dir / 'stitched_representation.tif'
    if verbose:
        with rasterio.open(local_tif) as src:
            print(f"\nGeoTIFF verified:")
            print(f"  Size   : {src.width}×{src.height} px")
            print(f"  Bands  : {src.count}  (expected 128)")
            print(f"  CRS    : {src.crs}")
            print(f"  Bounds : {src.bounds}")
            print(f"  Dtype  : {src.dtypes[0]}")
            print(f"  Size   : {local_tif.stat().st_size / 1e6:.1f} MB")

    # Copy to Drive
    drive_out = paths.OUTPUT_DIR / f'{area_name}'
    drive_out.mkdir(parents=True, exist_ok=True)

    # save with cloud cover threshold:
    dest = drive_out / f'TESSERA_{area_name}_{year}_t{str(max_cloud).zfill(3)}_embed.tif'
    print(f"\n[Export] Copying to Drive: {dest}")
    shutil.copy2(local_tif, dest)
    print(f"  ✓ Saved — {dest.stat().st_size / 1e6:.1f} MB")

    return dest

---
### Cell 15 — Cleanup

Removing all temporary files to prepare for the next run

In [ ]:
def worker_step_cleanup(paths: SimpleNamespace, verbose=False,
                        keep_download=False) -> None:
    """
    Removes all temporary files from TEMP_DIR after a successful pipeline run.
    Call after worker_step_export to free space before the next area/year.
    """
    if not keep_download:
      dirs_to_clean = [
          paths.TEMP_DIR / 's2_download_output',
          paths.TEMP_DIR / 's2_download_temp',
          paths.TEMP_DIR / 's1_download_output',
          paths.TEMP_DIR / 'stacked_output',
          paths.TEMP_DIR / 'retiled_d_pixel',
          paths.TEMP_DIR / 'representation_retiled',
          paths.TEMP_DIR / 'stitched_output',
      ]
    else:
      dirs_to_clean = [
          paths.TEMP_DIR / 'stacked_output',
          paths.TEMP_DIR / 'retiled_d_pixel',
          paths.TEMP_DIR / 'representation_retiled',
          paths.TEMP_DIR / 'stitched_output',
      ]

    print(f"\n[Cleanup] Removing temporary files...")
    for d in dirs_to_clean:
        if d.exists():
            shutil.rmtree(d)
            print(f"  ✓ Removed: {d}")
        else:
            print(f"  - Skipped: {d} (not found)")
    print("[Cleanup] Done.")


---
### Full Pipeline (steps 9-15)

By area and year (inner loop)

In [2]:
def run_worker_pipeline(name, params, year, roi_path, paths,
                        verbose: bool=False, max_cloud=20, chunksize=1024):
    """
    The primary orchestrator for a single area-year product.
    """
    print(f"\n{'='*60}")
    print(f"[WORKER] Pipeline execution start: {name} ({year} t{str(max_cloud).zfill(3)})")
    print(f"{'='*60}")

    # # Step 9: Download S1/S2 data
    # download_dirs = worker_step_download(roi_path, year, paths)
    s2_dir = worker_step_download_s2(roi_path, year, paths, max_cloud=max_cloud,
                                     chunksize=chunksize)
    s1_dir = worker_step_download_s1(roi_path, year, paths)

    # # Step 10: Stack data
    stacked_dir = worker_step_stack(s1_dir = s1_dir,
                                    s2_dir = s2_dir,
                                    paths  = paths)

    # # Step 11: Retile data
    retiled_dir = worker_step_retile(
        roi_path    = roi_path,
        stacked_dir = stacked_dir,
        paths       = paths,
        patch_size  = 40
    )

    # Step 12: Run Inference
    repr_dir = worker_step_inference(
        retiled_dir = retiled_dir,
        paths       = paths,
        ckpt_local  = paths.CKPT_LOCAL
    )

    # Step 13: Stitch patches
    stitched_dir = worker_step_stitch(
        roi_path    = roi_path,
        retiled_dir = retiled_dir,
        repr_dir    = repr_dir,
        paths       = paths
    )

    # Step 14: Final Export to GeoTIFF
    final_tif = worker_step_export(
        area_name    = name,
        year         = year,
        roi_path     = roi_path,
        stitched_dir = stitched_dir,
        paths        = paths,
        max_cloud    = max_cloud
    )

    # Step 15: Cleanup temporary files
    worker_step_cleanup(paths, )

    print(f"\n[WORKER] Pipeline execution complete: {name} ({year})")
    #print(f"[RESULT] Final GeoTIFF saved to: {final_tif.name}")
    print(f"{'='*60}\n")